In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# PROJECT: Fault Detection and Identification in LV Grids
# ============================================================
# This script establishes the normal operation baseline for
# the 5-node kite grid using:
#   1. Power Flow    — compute voltages and currents per timestep
#   2. State Estimation (SE) — estimate voltages via WLS
#   3. AR Baseline   — model I12 temporal behaviour via AR(1)
#
# All three methods are run over the synthetic dataset (500 t)
# generated from the original 13 lab measurements.
# ============================================================

np.random.seed(42)

# ============================================================
# PARAMETERS (same as labs)
# ============================================================
networkFactor = 100
cosPhi        = 0.95
sig_m         = 0.001   # noise on real measurements
sig_pm        = 0.5     # noise on pseudo-measurements
m             = 200     # Monte Carlo iterations for SE

# ============================================================
# LOAD DATA
# ============================================================

# Synthetic load data generated by extend_data.py (500 x 4)
# Columns: Node 1, Node 2, Node 3, Node 4  (Node 5 = slack)
data_synthetic = np.load('data/load_synthetic.npy')   # (500, 4)
data_real      = np.load('data/load_real.npy')         # (13,  4)

T = data_synthetic.shape[0]   # 500 timesteps
print(f"Loaded synthetic data: {T} timesteps x {data_synthetic.shape[1]} nodes\n")

# ============================================================
# ADMITTANCE MATRIX  (from lab — same grid)
# ============================================================
Net_Info = np.array([
    [1, 2, complex(0.01, -0.1)  * networkFactor],
    [1, 3, complex(0.02, -0.2)  * networkFactor],
    [2, 3, complex(0.03, -0.2)  * networkFactor],
    [3, 4, complex(0.03, -0.2)  * networkFactor],
    [4, 5, complex(0.02, -0.2)  * networkFactor],
], dtype=object)

nBus   = 5
SlackBus = 5

Y = np.zeros((nBus, nBus), dtype=complex)
for row in Net_Info:
    i, j, y = int(row[0])-1, int(row[1])-1, row[2]
    Y[i, i] += y
    Y[j, j] += y
    Y[i, j] -= y
    Y[j, i] -= y

# Reduced admittance matrix (slack bus removed)
Yl = np.delete(np.delete(Y, SlackBus-1, axis=0), SlackBus-1, axis=1)

# Line admittances (for SE and current computation)
y12 = -Y[0, 1]
y13 = -Y[0, 2]
y23 = -Y[1, 2]
y34 = -Y[2, 3]
y45 = -Y[3, 4]

print("Admittance matrix Y:\n", np.round(Y, 2), "\n")

# ============================================================
# SECTION 1 — POWER FLOW
# Compute true voltages and I12 for each synthetic timestep.
# This is the "ground truth" for the baseline.
# ============================================================
print("=" * 55)
print("SECTION 1 — POWER FLOW OVER SYNTHETIC DATA")
print("=" * 55)

# Convert loads to complex power injections (same as labs)
# P = -Load * exp(j * arccos(cosPhi))
P_synthetic = -data_synthetic * np.exp(1j * np.arccos(cosPhi))  # (500, 4)

# Storage
V_true   = np.zeros((T, nBus),  dtype=complex)   # voltages at all 5 buses
I12_true = np.zeros(T)                            # current magnitude on line 1-2

for t in range(T):
    I_inj = np.conj(P_synthetic[t, :])            # current injections (4 nodes)
    v_reduced = 1 + np.linalg.solve(Yl, I_inj)   # voltages at nodes 1-4
    V_true[t, :4] = v_reduced
    V_true[t,  4] = 1.0                           # slack bus voltage = 1 pu

    I12_complex = -Y[0, 1] * (V_true[t, 0] - V_true[t, 1])
    I12_true[t] = np.abs(I12_complex) * np.sign(np.real(I12_complex))

print(f"Power flow computed for {T} timesteps.")
print(f"I12 range: [{I12_true.min():.4f}, {I12_true.max():.4f}]")
print(f"V1 magnitude range: [{np.abs(V_true[:,0]).min():.4f}, {np.abs(V_true[:,0]).max():.4f}]\n")

# Plot: I12 over time
plt.figure(figsize=(12, 4))
plt.plot(I12_true, color='steelblue', linewidth=0.8, label='I12 (normal operation)')
plt.xlabel('Timestep')
plt.ylabel('Current I12 (p.u.)')
plt.title('Power Flow — Current I12 over Synthetic Normal Operation Data')
plt.legend()
plt.tight_layout()
plt.savefig('images/baseline_powerflow_I12.png', dpi=150)
plt.close()
print("Saved: images/baseline_powerflow_I12.png\n")

# ============================================================
# SECTION 2 — STATE ESTIMATION BASELINE
# Run WLS state estimation on each timestep using:
#   Real measurements:   I12, I54, V2, V5
#   Pseudo-measurements: I1, I2, I3, I4  (nodal injections)
# Same Hx and Wz as Lab 3.
# ============================================================
print("=" * 55)
print("SECTION 2 — STATE ESTIMATION BASELINE")
print("=" * 55)

# Build Hx matrix (same as Lab 3, Case 2b)
Hx = np.zeros((8, 5), dtype=complex)
Hx[0, 0] =  y12;  Hx[0, 1] = -y12          # I12
Hx[1, 3] = -y45;  Hx[1, 4] =  y45          # I54
Hx[2, 1] =  1                               # V2
Hx[3, 4] =  1                               # V5
Hx[4, :] =  Y[0, :]                         # I1 injection
Hx[5, :] =  Y[1, :]                         # I2 injection
Hx[6, :] =  Y[2, :]                         # I3 injection
Hx[7, :] =  Y[3, :]                         # I4 injection

# Weight matrix Wz (same as Lab 3)
Wz = np.zeros((8, 8))
np.fill_diagonal(Wz[:4, :4],   sig_m**-2)   # real measurements — high weight
np.fill_diagonal(Wz[4:8, 4:8], sig_pm**-2)  # pseudo-measurements — low weight

# Precompute WLS gain matrix (constant across timesteps — same grid)
G_se  = Hx.conj().T @ Wz @ Hx
G_inv = np.linalg.inv(G_se)

# Storage for SE results
V_est       = np.zeros((T, 5),  dtype=complex)   # estimated voltages
SE_residuals = np.zeros((T, 8), dtype=complex)   # measurement residuals

for t in range(T):
    I_inj = np.conj(P_synthetic[t, :])

    # Build measurement vector z
    z = np.zeros(8, dtype=complex)
    z[0] = -Y[0, 1] * (V_true[t, 0] - V_true[t, 1])   # I12
    z[1] = -Y[3, 4] * (V_true[t, 4] - V_true[t, 3])   # I54
    z[2] =  V_true[t, 1]                                # V2
    z[3] =  V_true[t, 4]                                # V5
    z[4:8] = I_inj                                       # pseudo: I1..I4

    # Add noise
    noise_m  = np.random.normal(0, sig_m,  4)
    noise_pm = np.random.normal(0, sig_pm, 4)
    z_noisy = z.copy()
    z_noisy[:4]  += noise_m
    z_noisy[4:8] += noise_pm

    # WLS estimation
    x_est = G_inv @ (Hx.conj().T @ Wz @ z_noisy)
    V_est[t, :]       = x_est
    SE_residuals[t, :] = z_noisy - Hx @ x_est

# SE residual norm per timestep (scalar anomaly signal)
SE_residual_norm = np.linalg.norm(np.abs(SE_residuals), axis=1)

print(f"SE residual norm — mean: {SE_residual_norm.mean():.4f},  std: {SE_residual_norm.std():.4f}")
print(f"Voltage estimation error |V_est - V_true| mean: {np.abs(V_est - V_true).mean():.6f}\n")

# Plot: SE residual norm over time
plt.figure(figsize=(12, 4))
plt.plot(SE_residual_norm, color='darkorange', linewidth=0.8, label='SE residual norm')
plt.axhline(SE_residual_norm.mean() + 3*SE_residual_norm.std(),
            color='red', linestyle='--', label='3σ threshold')
plt.xlabel('Timestep')
plt.ylabel('Residual norm')
plt.title('State Estimation — Residual Norm (Normal Operation Baseline)')
plt.legend()
plt.tight_layout()
plt.savefig('images/baseline_SE_residuals.png', dpi=150)
plt.close()
print("Saved: images/baseline_SE_residuals.png\n")

# ============================================================
# SECTION 3 — AR(1) BASELINE FOR I12
# Fit an AR(1) model on I12 from the synthetic data.
# Model: I12(t) = b0 + b1*I12(t-1) + b2*P1_wind(t) + eps
# P1 wind = real part of injection at node 1
# ============================================================
print("=" * 55)
print("SECTION 3 — AR(1) BASELINE FOR I12")
print("=" * 55)

# Wind injection at node 1 (real part)
P1_wind = np.real(P_synthetic[:, 0])

# AR(1) design matrix: [1, I12(t-1), P1_wind(t)]
# We use timesteps 1..T-1 to predict I12(t) from I12(t-1)
T_ar  = T - 1
X_ar  = np.ones((T_ar, 3))
X_ar[:, 1] = I12_true[:T_ar]    # I12(t-1)
X_ar[:, 2] = P1_wind[1:]        # P1_wind(t)
y_ar  = I12_true[1:]             # I12(t)  — target

# OLS estimation of AR coefficients
beta_ar = np.linalg.lstsq(X_ar, y_ar, rcond=None)[0]
print(f"AR(1) coefficients:")
print(f"  b0 (intercept):    {beta_ar[0]:.4f}")
print(f"  b1 (I12 lag-1):    {beta_ar[1]:.4f}")
print(f"  b2 (P1_wind):      {beta_ar[2]:.4f}\n")

# AR predictions and residuals
I12_ar_pred     = X_ar @ beta_ar
AR_residuals    = y_ar - I12_ar_pred

# Durbin-Watson statistic
dw_num = np.sum(np.diff(AR_residuals)**2)
dw_den = np.sum(AR_residuals**2)
DW     = dw_num / dw_den
print(f"Durbin-Watson statistic: {DW:.4f}  (2.0 = no autocorrelation)")
print(f"AR residuals — mean: {AR_residuals.mean():.6f},  std: {AR_residuals.std():.4f}\n")

# Plot: AR predictions vs true I12
plt.figure(figsize=(12, 4))
plt.plot(y_ar[:100],         color='steelblue', linewidth=1.0, label='True I12')
plt.plot(I12_ar_pred[:100],  color='tomato',    linewidth=1.0, linestyle='--', label='AR(1) prediction')
plt.xlabel('Timestep')
plt.ylabel('Current I12 (p.u.)')
plt.title('AR(1) Baseline — Predicted vs True I12 (first 100 timesteps)')
plt.legend()
plt.tight_layout()
plt.savefig('images/baseline_AR_prediction.png', dpi=150)
plt.close()

# Plot: AR residuals
plt.figure(figsize=(12, 4))
plt.plot(AR_residuals, color='purple', linewidth=0.8, label='AR residuals')
plt.axhline( 3*AR_residuals.std(), color='red', linestyle='--', label='±3σ threshold')
plt.axhline(-3*AR_residuals.std(), color='red', linestyle='--')
plt.xlabel('Timestep')
plt.ylabel('Residual')
plt.title('AR(1) Baseline — Residuals (Normal Operation)')
plt.legend()
plt.tight_layout()
plt.savefig('images/baseline_AR_residuals.png', dpi=150)
plt.close()
print("Saved: images/baseline_AR_prediction.png")
print("Saved: images/baseline_AR_residuals.png\n")

# ============================================================
# SECTION 4 — SAVE BASELINE FOR USE IN FAULT DETECTION
# ============================================================
print("=" * 55)
print("SECTION 4 — SAVING BASELINE")
print("=" * 55)

# Thresholds: mean + 3*std of each residual signal under normal operation
# These will be used in the fault detection stage to flag anomalies
threshold_SE = SE_residual_norm.mean() + 3 * SE_residual_norm.std()
threshold_AR = 3 * AR_residuals.std()

print(f"SE anomaly threshold (mean + 3σ): {threshold_SE:.4f}")
print(f"AR anomaly threshold (±3σ):       {threshold_AR:.4f}\n")

baseline = {
    'V_true':            V_true,          # (500, 5) complex voltages
    'I12_true':          I12_true,         # (500,)   true I12
    'V_est':             V_est,            # (500, 5) SE estimated voltages
    'SE_residuals':      SE_residuals,     # (500, 8) SE residuals
    'SE_residual_norm':  SE_residual_norm, # (500,)   scalar SE signal
    'AR_residuals':      AR_residuals,     # (499,)   AR residuals
    'beta_ar':           beta_ar,          # (3,)     AR coefficients
    'threshold_SE':      threshold_SE,     # scalar
    'threshold_AR':      threshold_AR,     # scalar
    'Hx':                Hx,              # (8,5) measurement matrix
    'Wz':                Wz,              # (8,8) weight matrix
    'Y':                 Y,               # (5,5) admittance matrix
}

np.save('data/baseline.npy', baseline, allow_pickle=True)
print("Baseline saved to data/baseline.npy")
print("\nNext step: load this baseline and inject faults on top of it.")
print("Use:  baseline = np.load('data/baseline.npy', allow_pickle=True).item()")

Loaded synthetic data: 500 timesteps x 4 nodes

Admittance matrix Y:
 [[ 3.-30.j -1.+10.j -2.+20.j  0. +0.j  0. +0.j]
 [-1.+10.j  4.-30.j -3.+20.j  0. +0.j  0. +0.j]
 [-2.+20.j -3.+20.j  8.-60.j -3.+20.j  0. +0.j]
 [ 0. +0.j  0. +0.j -3.+20.j  5.-40.j -2.+20.j]
 [ 0. +0.j  0. +0.j  0. +0.j -2.+20.j  2.-20.j]] 

SECTION 1 — POWER FLOW OVER SYNTHETIC DATA
Power flow computed for 500 timesteps.
I12 range: [-3.0106, 2.5636]
V1 magnitude range: [0.9044, 2.5099]

Saved: images/baseline_powerflow_I12.png

SECTION 2 — STATE ESTIMATION BASELINE
SE residual norm — mean: 0.7808,  std: 0.3372
Voltage estimation error |V_est - V_true| mean: 0.001837

Saved: images/baseline_SE_residuals.png

SECTION 3 — AR(1) BASELINE FOR I12
AR(1) coefficients:
  b0 (intercept):    0.2792
  b1 (I12 lag-1):    -0.0389
  b2 (P1_wind):      0.2651

Durbin-Watson statistic: 1.9866  (2.0 = no autocorrelation)
AR residuals — mean: -0.000000,  std: 0.3676

Saved: images/baseline_AR_prediction.png
Saved: images/baseline_AR